# COMM074 – Practical Business Analytics
## Group Notebook: Sections 1 & 2 – Data Loading, EDA, and Pre-processing

**Group Members:** Akash Chohan, Parth, Piyush, Nagashri
**Dataset:** IBM Transactions for Anti-Money Laundering (AML) – HI-Small
**Source:** https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml

### Overview
This master notebook covers the first two phases of the CRISP-DM lifecycle:
* **Section 1: Data Understanding & EDA:** Dataset loading, initial inspection, data type checking, summary statistics, and graphical analysis (histograms, boxplots, class distribution, correlation). *(Lead: Parth)*
* **Section 2: Data Pre-Processing:** Feature scaling, handling missing values, and addressing class imbalance using SMOTE. *(Lead: Nagashri)*
* **Finalization:** Data verification and export to highly optimized formats (Parquet/Numpy) for individual modeling. *(Lead: Akash)*

## 0. Install / Import Libraries

In [2]:
# Standard data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# Suppress minor warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# Consistent plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully.')

Libraries loaded successfully.


## 1. Dataset Loading

We use the **HI-Small** variant of the IBM AML dataset. 'HI' means a higher illicit (laundering) ratio, making it more suitable for training classification models at coursework scale.

**Instructions:** Place `HI_Small_Trans.csv` in the same directory as this notebook and update the `FILE_PATH` below if needed.

In [3]:
from google.colab import files
uploaded = files.upload()  # a button will appear to upload the CSV
FILE_PATH = 'HI_Small_Trans.csv'



KeyboardInterrupt: 

In [ ]:
import pandas as pd

# Using the exact name with the hyphen
FILE_PATH = 'HI-Small_Trans.csv'

df = pd.read_csv(
    FILE_PATH,
    dtype={
        'Amount Received': float,
        'Amount Paid': float
    },
    na_values=['', 'NA', 'N/A', 'null']
)

print(f'Dataset loaded. Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')

## 2. Initial Inspection

In [ ]:
# --- 2.1 Column renaming ---
# 'Account' and 'Account.1' are a Pandas artefact from duplicate column names in the CSV.
# Rename immediately for clarity throughout the notebook.
df.rename(columns={
    'Account':   'From Account',
    'Account.1': 'To Account'
}, inplace=True)

print('Columns after rename:')
print(df.columns.tolist())

In [ ]:
# --- 2.2 First look at the data ---
df.head(10)

In [ ]:
df.info()

In [ ]:
# --- 2.4 Missing value audit ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Missing value report:')
print(missing_report[missing_report['Missing Count'] > 0] if missing.sum() > 0 else 'No missing values found.')

In [ ]:
# --- 2.5 Duplicate row check ---
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes:,} ({n_dupes / len(df) * 100:.3f}% of dataset)')

In [ ]:
# --- Clean Duplicates ---
df = df.drop_duplicates()
print(f"Duplicates removed. Clean dataset shape: {df.shape[0]:,} rows\n")

## 3. Timestamp Parsing & Feature Extraction

The `Timestamp` column is stored as a string. We parse it to datetime and extract time-based features.
These derived features can be informative for modelling – e.g. laundering activity may cluster at unusual hours.

In [ ]:
# Parse Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Extract useful temporal features
df['Hour']       = df['Timestamp'].dt.hour
df['DayOfWeek']  = df['Timestamp'].dt.dayofweek   # 0=Monday, 6=Sunday
df['DayOfMonth'] = df['Timestamp'].dt.day

print('Timestamp range:')
print(f"  From: {df['Timestamp'].min()}")
print(f"  To:   {df['Timestamp'].max()}")
print(f"  Span: {(df['Timestamp'].max() - df['Timestamp'].min()).days} days")

## 4. Summary Statistics

In [ ]:
# --- 4.1 Numeric columns summary ---
# .T transposes for easier reading with many stats
df[['Amount Paid', 'Amount Received']].describe().T.round(2)

In [ ]:
# --- 4.2 Categorical columns summary ---
cat_cols = ['Payment Format', 'Payment Currency', 'Receiving Currency']
for col in cat_cols:
    print(f'\n{col} – unique values: {df[col].nunique()}')
    print(df[col].value_counts(normalize=True).mul(100).round(2).to_string())

In [ ]:
# --- 4.3 Target variable distribution ---
# Is Laundering: 0 = legitimate, 1 = laundering
target_counts = df['Is Laundering'].value_counts()
target_pct    = df['Is Laundering'].value_counts(normalize=True).mul(100).round(4)

print('Target class distribution:')
print(pd.DataFrame({'Count': target_counts, 'Percentage (%)': target_pct}))
print(f'\nClass imbalance ratio – 1 laundering per {int(target_counts[0]/target_counts[1]):,} legitimate transactions')

## 5. Exploratory Visualisations

In [ ]:
# --- 5.1 Target class bar chart ---
fig, ax = plt.subplots(figsize=(5, 4))
labels = ['Legitimate (0)', 'Laundering (1)']
colors = ['#4C72B0', '#DD8452']
bars = ax.bar(labels, target_counts.values, color=colors, edgecolor='white', linewidth=0.8)

# Annotate bars with count and percentage
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(target_counts)*0.01,
            f'{count:,}\n({pct:.2f}%)', ha='center', va='bottom', fontsize=9)

ax.set_title('Class Distribution: Legitimate vs Laundering Transactions', fontsize=11, pad=12)
ax.set_ylabel('Transaction Count')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig('class_distribution.png', bbox_inches='tight')
plt.show()
print('NOTE: Severe class imbalance – justifies use of SMOTE in pre-processing.')

In [ ]:
# --- 5.2 Amount Paid distribution (log scale) ---
# Log scale is used because transaction amounts are highly right-skewed
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, label in enumerate(['Legitimate', 'Laundering']):
    subset = df[df['Is Laundering'] == i]['Amount Paid']
    axes[i].hist(subset, bins=60, color=colors[i], edgecolor='white', linewidth=0.4, log=True)
    axes[i].set_title(f'Amount Paid – {label} Transactions', fontsize=10)
    axes[i].set_xlabel('Amount Paid')
    axes[i].set_ylabel('Frequency (log scale)')
    axes[i].xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.suptitle('Distribution of Transaction Amounts (Log Y-scale)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('amount_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.3 Boxplot: Amount Paid by class ---
fig, ax = plt.subplots(figsize=(6, 5))
plot_data = [
    df[df['Is Laundering'] == 0]['Amount Paid'].dropna(),
    df[df['Is Laundering'] == 1]['Amount Paid'].dropna()
]
bp = ax.boxplot(plot_data, labels=['Legitimate', 'Laundering'],
                patch_artist=True, showfliers=False)  # showfliers=False keeps plot readable
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title('Transaction Amount by Class (Outliers Hidden)', fontsize=11)
ax.set_ylabel('Amount Paid')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('boxplot_amount.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.4 Payment Format vs Laundering ---
# Stacked bar showing proportion of laundering within each payment format
fmt_counts = df.groupby(['Payment Format', 'Is Laundering']).size().unstack(fill_value=0)
fmt_pct = fmt_counts.div(fmt_counts.sum(axis=1), axis=0) * 100

ax = fmt_pct.plot(kind='bar', stacked=True, figsize=(9, 5),
                  color=colors, edgecolor='white', linewidth=0.5)
ax.set_title('Payment Format: Proportion of Legitimate vs Laundering Transactions', fontsize=11)
ax.set_xlabel('Payment Format')
ax.set_ylabel('Percentage (%)')
ax.legend(['Legitimate', 'Laundering'], bbox_to_anchor=(1.01, 1))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('payment_format_laundering.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.5 Transactions by Hour of Day ---
# Hypothesis: laundering may occur at unusual hours (e.g. late night)
fig, ax = plt.subplots(figsize=(10, 4))
for i, label in enumerate(['Legitimate', 'Laundering']):
    subset = df[df['Is Laundering'] == i]
    hourly = subset.groupby('Hour').size()
    # Normalise so the two very different class sizes are comparable
    hourly_norm = hourly / hourly.sum() * 100
    ax.plot(hourly_norm.index, hourly_norm.values, label=label, color=colors[i], marker='o', markersize=4)

ax.set_title('Transaction Volume by Hour of Day (% within class)', fontsize=11)
ax.set_xlabel('Hour of Day (0–23)')
ax.set_ylabel('% of Class Transactions')
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.savefig('hourly_pattern.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.6 Currency mismatch flag ---
# Transactions where Payment Currency != Receiving Currency may indicate cross-border layering
df['Currency Mismatch'] = (df['Payment Currency'] != df['Receiving Currency']).astype(int)

mismatch_by_class = df.groupby('Is Laundering')['Currency Mismatch'].mean() * 100
print('Currency mismatch rate by class:')
print(mismatch_by_class.rename({0: 'Legitimate', 1: 'Laundering'}).round(2).to_string())

fig, ax = plt.subplots(figsize=(5, 4))
mismatch_by_class.rename({0: 'Legitimate', 1: 'Laundering'}).plot(
    kind='bar', ax=ax, color=colors, edgecolor='white'
)
ax.set_title('Currency Mismatch Rate by Class (%)', fontsize=11)
ax.set_ylabel('% of Transactions')
ax.set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('currency_mismatch.png', bbox_inches='tight')
plt.show()

In [ ]:
# --- 5.7 Correlation heatmap (numeric features only) ---
# Helps identify redundant or strongly related features before modelling
numeric_cols = ['Amount Paid', 'Amount Received', 'Hour', 'DayOfWeek',
                'DayOfMonth', 'Currency Mismatch', 'Is Laundering']
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix – Numeric Features', fontsize=11)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

## 6. Data Dictionary & Field Assessment

| Column | Type | Description | Inclusion Decision |
|--------|------|-------------|--------------------|
| Timestamp | datetime | Date and time of transaction | Derived features (Hour, DayOfWeek, DayOfMonth) to be included; raw Timestamp excluded |
| From Bank | int64 | Sending bank ID | Include (kept as numeric identifier) |
| From Account | object | Sending account ID | Exclude (too high cardinality; potential data leakage) |
| To Bank | int64 | Receiving bank ID | Include (kept as numeric identifier) |
| To Account | object | Receiving account ID | Exclude (too high cardinality) |
| Amount Paid | float64 | Amount sent by source | Include (scale required) |
| Payment Currency | object | Currency of payment | Include (one-hot encode) |
| Amount Received | float64 | Amount received at destination | Include (scale required) |
| Receiving Currency | object | Currency at destination | Include (one-hot encode) |
| Payment Format | object | Transaction type (wire, ACH, etc.) | Include (one-hot encode) |
| Is Laundering | int64 | **Target** – 1=laundering, 0=legitimate | Target variable |
| Currency Mismatch | int64 | **Derived** – 1 if currencies differ | Include |
| Hour | int64 | **Derived** – Hour of transaction | Include |
| DayOfWeek | int64 | **Derived** – Day of week (0=Mon) | Include |
| DayOfMonth | int64 | **Derived** – Day of month | Include |

## 7. EDA Summary & Handoff to Pre-processing

Key findings to communicate to the group:

1. **Class imbalance is severe** – laundering accounts for ~0.1% of transactions. Nagashri's SMOTE step is absolutely essential to ensure the models learn the minority class.
2. **Amount distributions are heavily right-skewed** – `StandardScaler` is required before modelling to handle the extreme outlier transaction values.
3. **Payment Format is highly informative** – specifically, **ACH** transactions show disproportionately high laundering rates. One-Hot Encoding is required for this categorical feature.
4. **Currency Mismatch** is a useful engineered binary feature – laundering transactions have a significantly higher mismatch rate.
5. **Feature Exclusion & Retention** – Account IDs have extremely high cardinality and risk data leakage, so they must be dropped. Bank IDs will be kept as raw numeric identifiers to avoid memory crashes from excessive encoding.
6. **No missing values detected** – simplifies the pre-processing pipeline.

The dataset is now ready for Nagashri to apply pre-processing (dropping columns, feature engineering, scaling, encoding, and SMOTE).

In [ ]:
# --- Save the lightly-processed dataframe for handoff ---
# This retains raw columns + derived features, before encoding/scaling
# Nagashri will apply the full pre-processing pipeline on top of this
df.to_csv('aml_after_eda.csv', index=False)
print('Saved aml_after_eda.csv – ready for pre-processing stage.')

In [ ]:
from google.colab import files

# Define the filename for the group's handoff
export_filename = 'aml_preprocessed_ready_for_smote.csv'

# Save the transformed dataframe to a CSV file in the Colab environment
df.to_csv(export_filename, index=False)

# Trigger the browser download to your local machine
files.download(export_filename)

NAGASHRI SMOTE



In [ ]:
from google.colab import files
uploaded = files.upload()   # pick aml_preprocessed_ready_for_smote.csv

In [ ]:
df = pd.read_csv('aml_preprocessed_ready_for_smote.csv')

Quick check on data


In [ ]:
print(f'Shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head()

confirm how many bool columns we have

In [ ]:
print(f'Shape: {df.shape}')
print(f'Target distribution:')
print(df['Is Laundering'].value_counts())
print(f'Laundering rate: {df["Is Laundering"].mean()*100:.4f}%')
print()
print('Data types:')
print(df.dtypes.value_counts())

Convert booleans to intergers

In [ ]:
bool_cols = df.select_dtypes(include='bool').columns.tolist()
print(f'Converting {len(bool_cols)} boolean columns to int (0/1)...')
df[bool_cols] = df[bool_cols].astype(int)
print('Done. Sample dtypes after conversion:')
print(df.dtypes.head(10))

Imports for splitting and SMOTE

In [ ]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import scipy.sparse as sp
import joblib

RANDOM_STATE = 42

Separate X and Y, then split

In [ ]:
y = df['Is Laundering'].astype(int)
X = df.drop(columns=['Is Laundering'])

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {X_train.shape}  |  Laundering rate: {y_train.mean()*100:.4f}%')
print(f'Test:  {X_test.shape}   |  Laundering rate: {y_test.mean()*100:.4f}%')

Apply SMOTE (training only)

In [ ]:
print('Before SMOTE (train):')
print(y_train.value_counts())

smote = SMOTE(
    random_state=RANDOM_STATE,
    k_neighbors=5,
    sampling_strategy=0.1   # minority becomes 10% of majority size
)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('\nAfter SMOTE (train):')
print(pd.Series(y_train_bal).value_counts())
print(f'\nBalanced training shape: {X_train_bal.shape}')

Saving outputs to the drive

In [ ]:
import numpy as np

X_train_bal.to_parquet('X_train_balanced.parquet')
X_test.to_parquet('X_test_processed.parquet')
np.save('y_train_balanced.npy', np.asarray(y_train_bal))
np.save('y_test.npy', np.asarray(y_test))

print('Saved to shared Drive folder:')
for f in ['X_train_balanced.parquet', 'X_test_processed.parquet',
          'y_train_balanced.npy', 'y_test.npy']:
    print(f'  • {f}')

import pandas as pd
import numpy as np

X_train = pd.read_parquet('X_train_balanced.parquet')
X_test  = pd.read_parquet('X_test_processed.parquet')
y_train = np.load('y_train_balanced.npy')
y_test  = np.load('y_test.npy')

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


LOAD THIS SNIPPET FOR MODELLING NOTEBOOK


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ==========================================
# FINALIZATION: DATA VERIFICATION
# ==========================================
import pandas as pd
import numpy as np

# Load the saved datasets to verify integrity
X_train_check = pd.read_parquet('/content/drive/MyDrive/COM074_Group_Project/X_train_balanced.parquet')
y_train_check = np.load('/content/drive/MyDrive/COM074_Group_Project/y_train_balanced.npy')
X_test_check = pd.read_parquet('/content/drive/MyDrive/COM074_Group_Project/X_test_processed.parquet')
y_test_check = np.load('/content/drive/MyDrive/COM074_Group_Project/y_test.npy')

print("--- Data Shape Verification ---")
print(f"Training Features: {X_train_check.shape}")
print(f"Training Labels:   {y_train_check.shape}")
print(f"Testing Features:  {X_test_check.shape}")
print(f"Testing Labels:    {y_test_check.shape}")

# Verify Class Balance
unique, counts = np.unique(y_train_check, return_counts=True)
balance_dict = dict(zip(unique, counts))
print(f"\n--- Class Balance (SMOTE) ---")
print(f"0 (Legitimate): {balance_dict.get(0, 0)}")
print(f"1 (Fraudulent): {balance_dict.get(1, 0)}")

# Verify Class Balance (Updated for 10% Strategy)
if balance_dict.get(1, 0) == int(balance_dict.get(0, 0) * 0.1):
    print("\n✅ SUCCESS: Classes are balanced to the target 10% ratio. Data is ready for Section 3 Modelling.")
else:
    print("\n⚠️ WARNING: Classes do not match the 10% target. Check SMOTE configuration.")

--- Data Shape Verification ---
Training Features: (4464378, 42)
Training Labels:   (4464378,)
Testing Features:  (1015668, 42)
Testing Labels:    (1015668,)

--- Class Balance (SMOTE) ---
0 (Legitimate): 4058526
1 (Fraudulent): 405852

✅ SUCCESS: Classes are balanced to the target 10% ratio. Data is ready for Section 3 Modelling.
